Task 1: Analyze and Modify the 8-Puzzle Code

In [3]:
import numpy as np

def coordinates(puzzle):
    pos = np.array(range(9))
    for i, x in enumerate(puzzle):
        pos[x] = i
    return pos

def misplaced_tiles(curr, goal):
    return np.sum(curr != goal) - (1 if 0 in curr else 0)

def evaluate_misplaced(puzzle, goal):
    steps = np.array([('up', [0, 1, 2], -3), ('down', [6, 7, 8], 3),
                      ('left', [0, 3, 6], -1), ('right', [2, 5, 8], 1)],
                     dtype=[('move', 'U5'), ('position', list), ('head', int)])

    dtstate = [('puzzle', list), ('parent', int), ('gn', int), ('hn', int)]
    costg = coordinates(goal)
    parent = -1
    gn = 0
    hn = misplaced_tiles(np.array(puzzle), np.array(goal))
    state = np.array([(puzzle, parent, gn, hn)], dtype=dtstate)

    dtpriority = [('position', int), ('fn', int)]
    priority = np.array([(0, hn)], dtype=dtpriority)

    visited = []

    while len(priority) > 0:
        priority = np.sort(priority, kind='mergesort', order=['fn', 'position'])
        position, fn = priority[0]
        priority = np.delete(priority, 0, 0)

        curr_puzzle, parent, gn, hn = state[position]
        visited.append(curr_puzzle)

        if np.array_equal(curr_puzzle, goal):
            return state, visited

        blank = int(np.where(np.array(curr_puzzle) == 0)[0].item())

        for move in steps:
            if blank not in move['position']:
                new_blank = blank + move['head']
                new_puzzle = list(curr_puzzle)
                new_puzzle[blank], new_puzzle[new_blank] = new_puzzle[new_blank], new_puzzle[blank]

                if new_puzzle not in visited:
                    new_gn = gn + 1
                    new_hn = misplaced_tiles(np.array(new_puzzle), np.array(goal))
                    new_fn = new_gn + new_hn

                    state = np.append(state, np.array([(new_puzzle, position, new_gn, new_hn)], dtype=dtstate), axis=0)
                    priority = np.append(priority, np.array([(len(state) - 1, new_fn)], dtype=dtpriority), axis=0)

    return state, visited

initial_puzzle = [2, 8, 3, 1, 6, 4, 7, 0, 5]
goal_state = [1, 2, 3, 8, 0, 4, 7, 6, 5]

final_state, path_visited = evaluate_misplaced(initial_puzzle, goal_state)
print("Goal Reached. Total states explored:", len(path_visited))

Goal Reached. Total states explored: 7


Task 2: Implement A* for Graph Shortest Path

In [2]:
import heapq

def a_star_graph(graph, heuristics, start, goal):
    open_list = [(heuristics[start], 0, start, [])]
    visited = set()

    while open_list:
        (f, g, current, path) = heapq.heappop(open_list)

        if current in visited:
            continue

        path = path + [current]

        if current == goal:
            return path, g

        visited.add(current)

        for neighbor, cost in graph[current].items():
            if neighbor not in visited:
                new_g = g + cost
                new_f = new_g + heuristics[neighbor]
                heapq.heappush(open_list, (new_f, new_g, neighbor, path))

    return None, float('inf')

graph_data = {
    'A': {'B': 1, 'C': 4},
    'B': {'A': 1, 'C': 2, 'D': 5},
    'C': {'A': 4, 'B': 2, 'D': 1},
    'D': {'B': 5, 'C': 1}
}

heuristic_values = {
    'A': 3,
    'B': 2,
    'C': 1,
    'D': 0
}

shortest_path, total_cost = a_star_graph(graph_data, heuristic_values, 'A', 'D')
print(f"Shortest Path: {shortest_path}")
print(f"Total Cost: {total_cost}")

Shortest Path: ['A', 'B', 'C', 'D']
Total Cost: 4
